In [1]:
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 


pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome']

predict_n = 26
max_epiweek = 26
    
boxcox = False

TEST_YEAR = 2023


In [2]:
batch_size = 2
epochs = 250
media =True
verbose = 1
doenca = 'dengue'
min_delta = 0.0
patience= 25
lr = 2e-4
min_year = 2015
model_name = f'base_{max_epiweek}_{TEST_YEAR}'

In [3]:
df = prep.load_cases_data(filename= f'./data/{doenca}.csv.gz')

enso = None
enso_neutro = None

In [4]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    X_train, y_train, X_future, norm_values, norm_enso =  prep.generate_regional_train_samples(df_reg, enso, 
                                                                                    TEST_YEAR,
                                                                                    max_epiweek = max_epiweek,
                                                                                    columns_to_normalize = columns_to_normalize,
                                                                                    min_year = min_year, 
                                                                                    boxcox = boxcox,
                                                                                    media = media)

    model = mc.LSTMLogNormalModel(hidden=64, features=len(columns_to_normalize) + 1, 
                        predict_n=52 - max_epiweek)
    
    #model_path = f'./saved_models/trained_dengue_{region}_{TEST_YEAR-1}_base.pt'
    #model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    #model.to(device)  
    
    trained_model, hist = mc.train_model(

            model=model,

            X_train=X_train,

            X_future=X_future,

            Y_train=y_train,

            label=label,

            batch_size=batch_size,

            epochs=epochs,

            lr=lr,

            patience=patience,

            verbose=verbose,

            doenca=doenca
)


Sul
Epoch 1/250 | Train: 0.5561 | Val: 0.4704
Epoch 2/250 | Train: 0.3138 | Val: 0.1867
Epoch 3/250 | Train: 0.1494 | Val: 0.1232
Epoch 4/250 | Train: 0.1007 | Val: 0.1000
Epoch 5/250 | Train: 0.0821 | Val: 0.0876
Epoch 6/250 | Train: 0.0674 | Val: 0.0803
Epoch 7/250 | Train: 0.0602 | Val: 0.0758
Epoch 8/250 | Train: 0.0536 | Val: 0.0728
Epoch 9/250 | Train: 0.0486 | Val: 0.0706
Epoch 10/250 | Train: 0.0454 | Val: 0.0691
Epoch 11/250 | Train: 0.0442 | Val: 0.0679
Epoch 12/250 | Train: 0.0417 | Val: 0.0671
Epoch 13/250 | Train: 0.0400 | Val: 0.0665
Epoch 14/250 | Train: 0.0400 | Val: 0.0659
Epoch 15/250 | Train: 0.0411 | Val: 0.0655
Epoch 16/250 | Train: 0.0389 | Val: 0.0648
Epoch 17/250 | Train: 0.0366 | Val: 0.0639
Epoch 18/250 | Train: 0.0364 | Val: 0.0610
Epoch 19/250 | Train: 0.0360 | Val: 0.0563
Epoch 20/250 | Train: 0.0350 | Val: 0.0555
Epoch 21/250 | Train: 0.0362 | Val: 0.0555
Epoch 22/250 | Train: 0.0319 | Val: 0.0544
Epoch 23/250 | Train: 0.0313 | Val: 0.0543
Epoch 24/250 | T